# ECG Project — Notebook 4  
## Five-Level ECG Risk Label Builder

This notebook converts the extracted hospital ECG findings into a provisional five-level ECG screening target:

| Level | Name |
|---:|---|
| 1 | Very Low |
| 2 | Low |
| 3 | Moderate |
| 4 | High |
| 5 | Very High |

It also standardizes the raw hospital findings into **25 possible ECG finding groups**.

> **Important:** These are provisional, rule-generated ECG screening labels for model development. They are not confirmed diagnoses, emergency decisions, or complete cardiovascular-risk assessments. The mapping should be reviewed by a qualified clinician before any real clinical use.


## Project output

For every eligible ECG report, this notebook creates:

- `risk_level`
- `risk_name`
- `risk_score`
- `possible_ecg_findings`
- `risk_reasons`
- five-level rule-based preview distribution
- preserved `dataset_split`

The preview distribution is **not model confidence**. Notebook 5 will generate learned probabilities using `predict_proba()` and probability calibration.


## 1. Install and import packages

In [ ]:
!pip install -q pandas numpy openpyxl

import io
import json
import re
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 150)
pd.set_option("display.max_rows", 150)


## 2. Upload the output ZIP created by Notebook 3

In [ ]:
from google.colab import files

uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise ValueError(
        "Upload ecg_dataset_outputs.zip created by Notebook 3."
    )

input_zip = Path(zip_names[0])
print("Selected file:", input_zip)


## 3. Extract and load the existing datasets

In [ ]:
input_dir = Path("/content/notebook3_outputs")

if input_dir.exists():
    shutil.rmtree(input_dir)

input_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(input_zip, "r") as archive:
    archive.extractall(input_dir)

required_files = [
    "model_ready_dataset.csv",
    "train_dataset.csv",
    "validation_dataset.csv",
    "test_dataset.csv",
    "reports_requiring_review.csv",
    "split_manifest.csv"
]

missing_files = [
    name for name in required_files
    if not (input_dir / name).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing required Notebook 3 files: {missing_files}"
    )

model_ready = pd.read_csv(
    input_dir / "model_ready_dataset.csv"
)
train_original = pd.read_csv(
    input_dir / "train_dataset.csv"
)
validation_original = pd.read_csv(
    input_dir / "validation_dataset.csv"
)
test_original = pd.read_csv(
    input_dir / "test_dataset.csv"
)
review_original = pd.read_csv(
    input_dir / "reports_requiring_review.csv"
)
split_manifest = pd.read_csv(
    input_dir / "split_manifest.csv"
)

print("Model-ready reports:", len(model_ready))
print("Train:", len(train_original))
print("Validation:", len(validation_original))
print("Test:", len(test_original))
print("Existing review reports:", len(review_original))


## 4. Define the 25 standardized possible ECG finding groups

The levels below are project-development labels, not universal clinical severity statements.

Technical findings such as `Lead Off` and `Poor-quality Data` are treated as **quality-control problems**, not cardiac-risk classes.


In [ ]:
RISK_LEVEL_NAMES = {
    1: "Very Low",
    2: "Low",
    3: "Moderate",
    4: "High",
    5: "Very High"
}

FINDING_CATALOG = pd.DataFrame([
    # Level 1
    (1, "Normal ECG", 1),
    (2, "Normal Sinus Rhythm", 1),
    (3, "Mild Sinus Arrhythmia", 1),

    # Level 2
    (4, "Sinus Bradycardia", 2),
    (5, "Sinus Tachycardia", 2),
    (6, "Short PR Interval", 2),
    (7, "Axis Deviation", 2),
    (8, "Poor R-Wave Progression", 2),
    (9, "Atrial Abnormality", 2),

    # Level 3
    (10, "T-Wave Abnormality", 3),
    (11, "ST-T Abnormality", 3),
    (12, "Left Ventricular High Voltage / LVH", 3),
    (13, "Premature Atrial Contraction", 3),
    (14, "Isolated Premature Ventricular Contraction", 3),
    (15, "First-Degree AV Block", 3),
    (16, "Incomplete Bundle Branch Block", 3),
    (17, "Low Voltage", 3),
    (18, "Prolonged QT / QTc", 3),

    # Level 4
    (19, "Frequent PVC / Bigeminy / Runs", 4),
    (20, "Complete Bundle or Fascicular Block", 4),
    (21, "Atrial Fibrillation / Flutter", 4),
    (22, "Suspected Ischemic ST Change", 4),
    (23, "Suspected or Prior Myocardial Infarction", 4),

    # Level 5
    (24, "Ventricular Tachyarrhythmia", 5),
    (25, "Critical Rhythm / Acute MI Pattern", 5)
], columns=["finding_id", "possible_ecg_finding", "base_risk_level"])

FINDING_CATALOG["risk_name"] = FINDING_CATALOG[
    "base_risk_level"
].map(RISK_LEVEL_NAMES)

display(FINDING_CATALOG)


## 5. Finding-recognition rules

Rules are applied to the complete `machine_findings_text`, not only to `primary_label`.

The order matters: more specific rules are checked before broader rules.


In [ ]:
TECHNICAL_QUALITY_PATTERNS = [
    r"\bpoor[- ]?quality data\b",
    r"\blead off\b",
    r"\belectrode(?:s)?\b.*\breversal\b",
    r"\breversal of\b.*\belectrode"
]

FINDING_RULES = [
    # Level 5
    {
        "finding": "Ventricular Tachyarrhythmia",
        "level": 5,
        "patterns": [
            r"\bventricular tachycardia\b",
            r"\bventricular fibrillation\b",
            r"\btorsades\b"
        ]
    },
    {
        "finding": "Critical Rhythm / Acute MI Pattern",
        "level": 5,
        "patterns": [
            r"\basystole\b",
            r"\bcomplete (?:atrioventricular|av|heart) block\b",
            r"\bthird[- ]degree (?:atrioventricular|av) block\b",
            r"\bacute myocardial infarction\b",
            r"\bstemi\b"
        ]
    },

    # Level 4
    {
        "finding": "Frequent PVC / Bigeminy / Runs",
        "level": 4,
        "patterns": [
            r"\bfrequent premature ventricular",
            r"\bpremature ventricular contraction bigeminy\b",
            r"\bventricular bigeminy\b",
            r"\bruns? of premature ventricular",
            r"\bventricular couplets?\b",
            r"\bventricular triplets?\b"
        ]
    },
    {
        "finding": "Complete Bundle or Fascicular Block",
        "level": 4,
        "patterns": [
            r"\bcomplete right bundle branch block\b",
            r"\bcomplete left bundle branch block\b",
            r"\bleft anterior hemi block\b",
            r"\bleft posterior hemi block\b",
            r"\bfascicular block\b"
        ]
    },
    {
        "finding": "Atrial Fibrillation / Flutter",
        "level": 4,
        "patterns": [
            r"\batrial fibrillation\b",
            r"\batrial flutter\b"
        ]
    },
    {
        "finding": "Suspected Ischemic ST Change",
        "level": 4,
        "patterns": [
            r"\bmarked st elevation\b",
            r"\bmiddle st depression\b",
            r"\bmarked st depression\b",
            r"\bsuspect(?:ed)? ischem",
            r"\bmyocardial ischem"
        ]
    },
    {
        "finding": "Suspected or Prior Myocardial Infarction",
        "level": 4,
        "patterns": [
            r"\bsuspect .*myocardial infarction",
            r"\bpossible .*myocardial infarction",
            r"\banterior myocardial infarction",
            r"\binferior myocardial infarction",
            r"\blateral myocardial infarction",
            r"\banteroseptal myocardial infarction",
            r"\babnormal [qQ] wave",
            r"\bqs wave"
        ]
    },

    # Level 3
    {
        "finding": "T-Wave Abnormality",
        "level": 3,
        "patterns": [
            r"\bt wave abnormality\b",
            r"\binverted t wave\b",
            r"\bnegative t wave\b",
            r"\bbiphasic t wave\b",
            r"\bflattened t wave\b",
            r"\blow t wave\b"
        ]
    },
    {
        "finding": "ST-T Abnormality",
        "level": 3,
        "patterns": [
            r"\bst[- ]t abnormality\b",
            r"\bnonspecific st",
            r"\bslight st elevation\b",
            r"\bslight st depression\b"
        ]
    },
    {
        "finding": "Left Ventricular High Voltage / LVH",
        "level": 3,
        "patterns": [
            r"\bhigh voltage.*left ventricle",
            r"\bleft ventricular hypertrophy"
        ]
    },
    {
        "finding": "Premature Atrial Contraction",
        "level": 3,
        "patterns": [
            r"\bpremature atrial contraction\b",
            r"\bpremature atrial complex\b"
        ]
    },
    {
        "finding": "Isolated Premature Ventricular Contraction",
        "level": 3,
        "patterns": [
            r"\bpremature ventricular contraction\b",
            r"\bpremature ventricular complex\b"
        ]
    },
    {
        "finding": "First-Degree AV Block",
        "level": 3,
        "patterns": [
            r"\bfirst[- ]degree atrioventricular block\b",
            r"\bfirst[- ]degree av block\b"
        ]
    },
    {
        "finding": "Incomplete Bundle Branch Block",
        "level": 3,
        "patterns": [
            r"\bincomplete right bundle branch block\b",
            r"\bincomplete left bundle branch block\b"
        ]
    },
    {
        "finding": "Low Voltage",
        "level": 3,
        "patterns": [
            r"\blow voltage\b"
        ]
    },
    {
        "finding": "Prolonged QT / QTc",
        "level": 3,
        "patterns": [
            r"\bprolonged qt interval\b",
            r"\bprolonged qtc\b"
        ]
    },

    # Level 2
    {
        "finding": "Sinus Bradycardia",
        "level": 2,
        "patterns": [
            r"\bsinus bradycardia\b"
        ]
    },
    {
        "finding": "Sinus Tachycardia",
        "level": 2,
        "patterns": [
            r"\bsinus tachycardia\b"
        ]
    },
    {
        "finding": "Short PR Interval",
        "level": 2,
        "patterns": [
            r"\bshort pr interval\b"
        ]
    },
    {
        "finding": "Axis Deviation",
        "level": 2,
        "patterns": [
            r"\bright axis deviation\b",
            r"\bleft axis deviation\b"
        ]
    },
    {
        "finding": "Poor R-Wave Progression",
        "level": 2,
        "patterns": [
            r"\bpoor [rR] wave progression\b"
        ]
    },
    {
        "finding": "Atrial Abnormality",
        "level": 2,
        "patterns": [
            r"\blarged ptfv1\b",
            r"\blarge ptfv1\b",
            r"\bprolonged p[- ]wave\b",
            r"\batrial abnormality\b"
        ]
    },

    # Level 1
    {
        "finding": "Mild Sinus Arrhythmia",
        "level": 1,
        "patterns": [
            r"\bsinus arrhythmia\b"
        ]
    },
    {
        "finding": "Normal ECG",
        "level": 1,
        "patterns": [
            r"\bnormal ecg\b"
        ]
    },
    {
        "finding": "Normal Sinus Rhythm",
        "level": 1,
        "patterns": [
            r"\bnormal sinus rhythm\b",
            r"^\s*sinus rhythm\s*$"
        ]
    }
]

def contains_pattern(text, patterns):
    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in patterns
    )

def normalize_finding_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = text.replace("***", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def detect_quality_problems(text):
    problems = []

    if re.search(r"\bpoor[- ]?quality data\b", text, re.IGNORECASE):
        problems.append("Poor-quality ECG data")

    if re.search(r"\blead off\b", text, re.IGNORECASE):
        problems.append("One or more ECG leads were disconnected")

    if re.search(
        r"(reversal of.*electrode|electrode.*reversal)",
        text,
        re.IGNORECASE
    ):
        problems.append("Possible electrode reversal")

    return list(dict.fromkeys(problems))

def detect_standardized_findings(text):
    detected = []

    # Apply high-severity and specific findings first.
    for rule in FINDING_RULES:
        if contains_pattern(text, rule["patterns"]):
            detected.append({
                "finding": rule["finding"],
                "level": rule["level"]
            })

    # Prevent broad isolated-PVC label when frequent/runs are present.
    frequent_name = "Frequent PVC / Bigeminy / Runs"
    isolated_name = "Isolated Premature Ventricular Contraction"

    names = {item["finding"] for item in detected}

    if frequent_name in names and isolated_name in names:
        detected = [
            item for item in detected
            if item["finding"] != isolated_name
        ]

    # Normal labels cannot coexist with recognized abnormal findings.
    abnormal_detected = any(item["level"] > 1 for item in detected)

    if abnormal_detected:
        detected = [
            item for item in detected
            if item["finding"] not in {
                "Normal ECG",
                "Normal Sinus Rhythm",
                "Mild Sinus Arrhythmia"
            }
        ]

    # Preserve order and remove duplicate findings.
    unique = {}
    for item in detected:
        unique[item["finding"]] = item

    return list(unique.values())


## 6. Parameter-based escalation rules

The hospital machine finding remains the main label source.

Measured parameters can increase concern when markedly abnormal. These thresholds are deliberately configurable and should be reviewed with the project's medical guide.

The code does **not** lower the level assigned by a recognized finding.


In [ ]:
PARAMETER_RULES = {
    "heart_rate_very_low": 40,
    "heart_rate_low": 50,
    "heart_rate_high": 100,
    "heart_rate_very_high": 150,
    "pr_short": 120,
    "pr_prolonged": 200,
    "qrs_wide": 120,
    "qrs_very_wide": 160,
    "qtc_borderline": 460,
    "qtc_marked": 500
}

def parameter_risk_assessment(row):
    level = 1
    reasons = []

    hr = row.get("heart_rate_bpm")
    pr = row.get("pr_interval_ms")
    qrs = row.get("qrs_duration_ms")
    qtc = row.get("qtc_bazett_ms")

    if pd.notna(hr):
        if hr < PARAMETER_RULES["heart_rate_very_low"]:
            level = max(level, 4)
            reasons.append(f"Markedly low heart rate ({hr:g} bpm)")
        elif hr < PARAMETER_RULES["heart_rate_low"]:
            level = max(level, 2)
            reasons.append(f"Low heart rate ({hr:g} bpm)")
        elif hr >= PARAMETER_RULES["heart_rate_very_high"]:
            level = max(level, 4)
            reasons.append(f"Markedly high heart rate ({hr:g} bpm)")
        elif hr > PARAMETER_RULES["heart_rate_high"]:
            level = max(level, 2)
            reasons.append(f"High heart rate ({hr:g} bpm)")

    if pd.notna(pr):
        if pr < PARAMETER_RULES["pr_short"]:
            level = max(level, 2)
            reasons.append(f"Short PR interval ({pr:g} ms)")
        elif pr > PARAMETER_RULES["pr_prolonged"]:
            level = max(level, 3)
            reasons.append(f"Prolonged PR interval ({pr:g} ms)")

    if pd.notna(qrs):
        if qrs >= PARAMETER_RULES["qrs_very_wide"]:
            level = max(level, 4)
            reasons.append(f"Markedly wide QRS ({qrs:g} ms)")
        elif qrs >= PARAMETER_RULES["qrs_wide"]:
            level = max(level, 3)
            reasons.append(f"Wide QRS ({qrs:g} ms)")

    if pd.notna(qtc):
        if qtc >= PARAMETER_RULES["qtc_marked"]:
            level = max(level, 4)
            reasons.append(f"Marked QTc prolongation ({qtc:g} ms)")
        elif qtc >= PARAMETER_RULES["qtc_borderline"]:
            level = max(level, 3)
            reasons.append(f"Prolonged or borderline QTc ({qtc:g} ms)")

    return level, reasons


## 7. Create a rule-based five-level preview distribution

This is included only to preview the final user-interface format.

It is generated from the assigned rule level and therefore must **not** be reported as trained-model confidence. Notebook 5 will replace these columns with calibrated model probabilities.


In [ ]:
PREVIEW_DISTRIBUTIONS = {
    1: [0.72, 0.20, 0.06, 0.015, 0.005],
    2: [0.15, 0.60, 0.19, 0.05, 0.01],
    3: [0.04, 0.17, 0.58, 0.17, 0.04],
    4: [0.01, 0.05, 0.19, 0.60, 0.15],
    5: [0.005, 0.015, 0.06, 0.20, 0.72]
}

def make_preview_distribution(level):
    values = PREVIEW_DISTRIBUTIONS[int(level)]
    return {
        f"rule_preview_probability_level_{index + 1}": value
        for index, value in enumerate(values)
    }

def expected_level_from_distribution(probabilities):
    return round(
        sum(
            level * probabilities[
                f"rule_preview_probability_level_{level}"
            ]
            for level in range(1, 6)
        ),
        3
    )


## 8. Assign provisional risk labels to every model-ready report

In [ ]:
def assign_risk(row):
    raw_findings = normalize_finding_text(
        row.get("machine_findings_text", "")
    )
    raw_primary = normalize_finding_text(
        row.get("primary_label", "")
    )
    rhythm = normalize_finding_text(
        row.get("rhythm", "")
    )

    combined_text = " | ".join(
        part for part in [raw_findings, raw_primary, rhythm]
        if part
    )

    quality_problems = detect_quality_problems(combined_text)
    detected = detect_standardized_findings(combined_text)

    finding_names = [
        item["finding"]
        for item in detected
    ]
    finding_levels = [
        item["level"]
        for item in detected
    ]

    # If only technical-quality findings are present, do not invent risk.
    non_quality_text = combined_text
    for pattern in TECHNICAL_QUALITY_PATTERNS:
        non_quality_text = re.sub(
            pattern,
            " ",
            non_quality_text,
            flags=re.IGNORECASE
        )

    has_recognized_finding = len(detected) > 0

    if quality_problems and not has_recognized_finding:
        return {
            "risk_eligible": False,
            "risk_level": np.nan,
            "risk_name": "Needs ECG Re-acquisition / Manual Review",
            "risk_score": np.nan,
            "possible_ecg_findings": "",
            "risk_reasons": " | ".join(quality_problems),
            "quality_control_flags": " | ".join(quality_problems),
            "unmapped_finding_text": combined_text
        }

    finding_level = max(finding_levels) if finding_levels else 1
    parameter_level, parameter_reasons = parameter_risk_assessment(row)

    final_level = max(finding_level, parameter_level)

    reasons = [
        f"{item['finding']} mapped to Level {item['level']}"
        for item in detected
        if item["level"] == finding_level
    ]

    reasons.extend(parameter_reasons)

    if quality_problems:
        reasons.extend(quality_problems)

    # If no finding was recognized, preserve it for review instead of
    # silently calling it normal.
    if not detected:
        return {
            "risk_eligible": False,
            "risk_level": np.nan,
            "risk_name": "Unmapped Finding — Manual Review",
            "risk_score": np.nan,
            "possible_ecg_findings": "",
            "risk_reasons": "No standardized finding rule matched",
            "quality_control_flags": " | ".join(quality_problems),
            "unmapped_finding_text": combined_text
        }

    preview = make_preview_distribution(final_level)
    expected_score = expected_level_from_distribution(preview)

    result = {
        "risk_eligible": True,
        "risk_level": int(final_level),
        "risk_name": RISK_LEVEL_NAMES[int(final_level)],
        "risk_score": expected_score,
        "possible_ecg_findings": " | ".join(finding_names),
        "risk_reasons": " | ".join(list(dict.fromkeys(reasons))),
        "quality_control_flags": " | ".join(quality_problems),
        "unmapped_finding_text": ""
    }

    result.update(preview)
    return result

risk_rows = model_ready.apply(assign_risk, axis=1)
risk_columns = pd.DataFrame(risk_rows.tolist())

risk_dataset = pd.concat(
    [
        model_ready.reset_index(drop=True),
        risk_columns.reset_index(drop=True)
    ],
    axis=1
)

print("All model-ready input reports:", len(risk_dataset))
print("Risk-eligible reports:", int(risk_dataset["risk_eligible"].sum()))
print(
    "Manual-review reports:",
    int((~risk_dataset["risk_eligible"]).sum())
)

display(
    risk_dataset[
        [
            "source_file",
            "primary_label",
            "machine_findings_text",
            "possible_ecg_findings",
            "risk_level",
            "risk_name",
            "risk_reasons"
        ]
    ].head(20)
)


## 9. Inspect the five-level distribution

In [ ]:
eligible_dataset = risk_dataset.loc[
    risk_dataset["risk_eligible"]
].copy()

manual_review_dataset = risk_dataset.loc[
    ~risk_dataset["risk_eligible"]
].copy()

risk_distribution = (
    eligible_dataset
    .groupby(["risk_level", "risk_name"])
    .size()
    .reset_index(name="report_count")
)

risk_distribution["percentage"] = (
    100 * risk_distribution["report_count"]
    / len(eligible_dataset)
).round(2)

display(risk_distribution)


## 10. Inspect the 25 possible ECG finding groups

In [ ]:
finding_rows = []

for _, row in eligible_dataset.iterrows():
    findings = [
        item.strip()
        for item in str(row["possible_ecg_findings"]).split("|")
        if item.strip()
    ]

    for finding in findings:
        finding_rows.append({
            "source_file": row["source_file"],
            "dataset_split": row.get("dataset_split", np.nan),
            "risk_level": row["risk_level"],
            "risk_name": row["risk_name"],
            "possible_ecg_finding": finding
        })

finding_long = pd.DataFrame(finding_rows)

finding_distribution = (
    finding_long["possible_ecg_finding"]
    .value_counts()
    .rename_axis("possible_ecg_finding")
    .reset_index(name="report_count")
    .merge(
        FINDING_CATALOG,
        on="possible_ecg_finding",
        how="right"
    )
    .fillna({"report_count": 0})
    .sort_values(
        ["base_risk_level", "finding_id"]
    )
)

finding_distribution["report_count"] = (
    finding_distribution["report_count"].astype(int)
)

display(finding_distribution)


## 11. Restore the fixed train, validation, and test split

Notebook 3 already created the split. This notebook joins by `sha256` and does not reshuffle reports.

Manual-review reports are removed from model-training outputs but retained in a separate file.


In [ ]:
split_lookup = split_manifest[
    ["sha256", "dataset_split"]
].drop_duplicates()

# Avoid duplicate dataset_split if one is already present.
eligible_dataset = eligible_dataset.drop(
    columns=["dataset_split"],
    errors="ignore"
).merge(
    split_lookup,
    on="sha256",
    how="left",
    validate="one_to_one"
)

manual_review_dataset = manual_review_dataset.drop(
    columns=["dataset_split"],
    errors="ignore"
).merge(
    split_lookup,
    on="sha256",
    how="left",
    validate="one_to_one"
)

if eligible_dataset["dataset_split"].isna().any():
    missing = eligible_dataset.loc[
        eligible_dataset["dataset_split"].isna(),
        "source_file"
    ].tolist()

    raise ValueError(
        f"Some reports lost their fixed split: {missing[:10]}"
    )

train_risk = eligible_dataset.loc[
    eligible_dataset["dataset_split"] == "train"
].copy()

validation_risk = eligible_dataset.loc[
    eligible_dataset["dataset_split"] == "validation"
].copy()

test_risk = eligible_dataset.loc[
    eligible_dataset["dataset_split"] == "test"
].copy()

print("Risk-eligible train rows:", len(train_risk))
print("Risk-eligible validation rows:", len(validation_risk))
print("Risk-eligible test rows:", len(test_risk))
print("Manual review rows:", len(manual_review_dataset))

# Confirm there is no overlap.
assert set(train_risk["sha256"]).isdisjoint(
    set(validation_risk["sha256"])
)
assert set(train_risk["sha256"]).isdisjoint(
    set(test_risk["sha256"])
)
assert set(validation_risk["sha256"]).isdisjoint(
    set(test_risk["sha256"])
)

print("Fixed-split leakage check passed.")


## 12. Check risk-level coverage in every split

In [ ]:
split_risk_distribution = pd.crosstab(
    eligible_dataset["risk_level"],
    eligible_dataset["dataset_split"]
).reindex(range(1, 6), fill_value=0)

split_risk_distribution["total"] = (
    split_risk_distribution.sum(axis=1)
)

display(split_risk_distribution)

missing_split_levels = []

for split_name in ["train", "validation", "test"]:
    for level in range(1, 6):
        count = int(
            (
                (eligible_dataset["dataset_split"] == split_name)
                & (eligible_dataset["risk_level"] == level)
            ).sum()
        )
        if count == 0:
            missing_split_levels.append(
                {
                    "dataset_split": split_name,
                    "risk_level": level,
                    "risk_name": RISK_LEVEL_NAMES[level]
                }
            )

missing_split_levels_df = pd.DataFrame(
    missing_split_levels
)

if len(missing_split_levels_df):
    print(
        "Warning: Some levels have no examples in one or more splits."
    )
    display(missing_split_levels_df)
else:
    print("Every split contains all five risk levels.")


## 13. Review unmapped and technical-quality reports

Do not force these reports into Low Risk.

They should be manually mapped, corrected, or excluded before training.


In [ ]:
review_columns = [
    "source_file",
    "primary_label",
    "machine_findings_text",
    "heart_rate_bpm",
    "pr_interval_ms",
    "qrs_duration_ms",
    "qtc_bazett_ms",
    "risk_name",
    "risk_reasons",
    "quality_control_flags",
    "unmapped_finding_text",
    "dataset_split"
]

display(
    manual_review_dataset[
        review_columns
    ].sort_values("risk_name")
)


## 14. Preview the intended final application output

In [ ]:
def preview_final_output(row):
    if not row["risk_eligible"]:
        print("ECG Risk Assessment")
        print("-------------------")
        print("Status: Manual review required")
        print("Reason:", row["risk_reasons"])
        return

    print("ECG Risk Assessment")
    print("-------------------")
    print(
        f"Predicted screening level: "
        f"{int(row['risk_level'])} — {row['risk_name']}"
    )
    print(
        f"Rule-based preview score: {row['risk_score']:.2f}/5"
    )
    print("\nFive-level preview distribution:")

    for level in range(1, 6):
        probability = row[
            f"rule_preview_probability_level_{level}"
        ]
        print(
            f"  Level {level} — "
            f"{RISK_LEVEL_NAMES[level]}: "
            f"{100 * probability:.1f}%"
        )

    print("\nPossible ECG findings:")
    for finding in row["possible_ecg_findings"].split("|"):
        finding = finding.strip()
        if finding:
            print(" •", finding)

    print("\nRisk reasons:")
    for reason in row["risk_reasons"].split("|"):
        reason = reason.strip()
        if reason:
            print(" •", reason)

    print("\nSelected parameters:")
    print(" Heart rate:", row["heart_rate_bpm"], "bpm")
    print(" PR interval:", row["pr_interval_ms"], "ms")
    print(" QRS duration:", row["qrs_duration_ms"], "ms")
    print(" QTc:", row["qtc_bazett_ms"], "ms")

    print(
        "\nDisclaimer: ECG-based screening output only; "
        "not a confirmed diagnosis."
    )

sample_row = eligible_dataset.sample(
    1,
    random_state=42
).iloc[0]

preview_final_output(sample_row)


## 15. Save Notebook 4 outputs

Outputs include:

- `risk_labeled_dataset.csv`
- `train_risk.csv`
- `validation_risk.csv`
- `test_risk.csv`
- `manual_review_risk_reports.csv`
- `finding_catalog_25.csv`
- `finding_distribution.csv`
- `risk_distribution.csv`
- `split_risk_distribution.csv`
- `risk_labeling_summary.json`
- `ecg_risk_label_outputs.zip`


In [ ]:
output_dir = Path("/content/ecg_risk_label_outputs")

if output_dir.exists():
    shutil.rmtree(output_dir)

output_dir.mkdir(parents=True, exist_ok=True)

eligible_dataset.to_csv(
    output_dir / "risk_labeled_dataset.csv",
    index=False
)

train_risk.to_csv(
    output_dir / "train_risk.csv",
    index=False
)

validation_risk.to_csv(
    output_dir / "validation_risk.csv",
    index=False
)

test_risk.to_csv(
    output_dir / "test_risk.csv",
    index=False
)

manual_review_dataset.to_csv(
    output_dir / "manual_review_risk_reports.csv",
    index=False
)

FINDING_CATALOG.to_csv(
    output_dir / "finding_catalog_25.csv",
    index=False
)

finding_distribution.to_csv(
    output_dir / "finding_distribution.csv",
    index=False
)

risk_distribution.to_csv(
    output_dir / "risk_distribution.csv",
    index=False
)

split_risk_distribution.to_csv(
    output_dir / "split_risk_distribution.csv"
)

if len(missing_split_levels_df):
    missing_split_levels_df.to_csv(
        output_dir / "missing_split_risk_levels.csv",
        index=False
    )

summary = {
    "input_model_ready_reports": int(len(model_ready)),
    "risk_eligible_reports": int(len(eligible_dataset)),
    "manual_review_reports": int(len(manual_review_dataset)),
    "standardized_finding_groups": int(len(FINDING_CATALOG)),
    "risk_levels": 5,
    "train_rows": int(len(train_risk)),
    "validation_rows": int(len(validation_risk)),
    "test_rows": int(len(test_risk)),
    "risk_level_counts": {
        str(int(row["risk_level"])): int(row["report_count"])
        for _, row in risk_distribution.iterrows()
    },
    "risk_level_names": {
        str(level): name
        for level, name in RISK_LEVEL_NAMES.items()
    },
    "probability_note": (
        "rule_preview_probability columns are interface previews, "
        "not trained-model confidence"
    ),
    "labeling_note": (
        "Provisional rule-generated ECG screening labels; "
        "clinical review is required before real-world use"
    )
}

with open(
    output_dir / "risk_labeling_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        indent=2,
        ensure_ascii=False
    )

archive_path = shutil.make_archive(
    "/content/ecg_risk_label_outputs",
    "zip",
    output_dir
)

print(json.dumps(summary, indent=2))
print("\nCreated:", archive_path)


## 16. Download the completed Notebook 4 package

In [ ]:
from google.colab import files

files.download("/content/ecg_risk_label_outputs.zip")


## Before Notebook 5

Check these files first:

1. `risk_distribution.csv`
2. `finding_distribution.csv`
3. `manual_review_risk_reports.csv`
4. `split_risk_distribution.csv`
5. `risk_labeling_summary.json`

Notebook 5 should train only after:

- the five levels have enough examples,
- the manual-review records are handled,
- and the mapping has been reviewed with the project guide or a clinician.

The trained model probabilities in Notebook 5 will replace the rule-based preview distribution.
